# Stereo-seq Reference18 K=6 Factor Validation

**Raw-expression validation and patient-level metrics for frozen per-chip Neighborhood NMF factors**

This report reuses the frozen K=6 NMF outputs. It does not refit RCTD or NMF, perform disease comparisons, or alter the 238 frozen lymphoid aggregates. All reported values are loaded dynamically from the completed analysis outputs.

## Reproducible render command

```bash
TS=$(date +%y%m%d%H%M%S)
PROJ='/mnt/d/Lung ST/codex agent/stereo_seq/stereo_seq'
/home/bozhu/miniconda3/envs/stereo/bin/jupyter nbconvert --to html --execute \
  "$PROJ/scripts/1103_k6_factor_downstream_report.ipynb" \
  --ExecutePreprocessor.timeout=600 \
  --output "1103_k6_factor_downstream_report_${TS}.html" \
  --output-dir "$PROJ/results" \
  > "$PROJ/logs/1103_k6_factor_downstream_report_${TS}.log" 2>&1
```

In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import Image, Markdown, display

relative_run_dir = Path('results/reanalysis/REFERENCE18_K6_FACTOR_DOWNSTREAM_20261013_003020')
project_root = next(
    root for root in [Path.cwd(), *Path.cwd().parents]
    if (root / relative_run_dir / 'COMPLETE.ok').exists()
)
run_dir = project_root / relative_run_dir
assert (run_dir / 'COMPLETE.ok').exists(), run_dir

manifest = pd.read_csv(run_dir / '00_INPUT_MANIFEST.tsv', sep='\t')
factor_definitions = pd.read_csv(run_dir / '01_FACTOR_DEFINITIONS.tsv', sep='\t')
patient_metrics = pd.read_csv(run_dir / '02_PATIENT_FACTOR_METRICS.tsv', sep='\t')
patient_expression = pd.read_csv(run_dir / '03_PATIENT_FACTOR_RAW_EXPRESSION_VALIDATION.tsv', sep='\t')
expression_summary = pd.read_csv(run_dir / '04_FACTOR_RAW_EXPRESSION_SUMMARY.tsv', sep='\t')
marker_audit = pd.read_csv(run_dir / '05_MARKER_PANEL_AUDIT.tsv', sep='\t')
factor_table = pd.read_csv(run_dir / '06_FINAL_FACTOR_ANALYSIS_TABLE.tsv', sep='\t')
validation = pd.read_csv(run_dir / '08_VALIDATION.tsv', sep='\t')
pd.set_option('display.max_rows', 200)
pd.set_option('display.max_columns', 50)

## Scope and validation status

In [ ]:
scope = pd.DataFrame({
    'Measure': ['Chips', 'Patients', 'Factors', 'Evaluable neighborhoods', 'Patient-factor rows', 'Validation checks passed'],
    'Value': [
        factor_table['chip'].nunique(),
        patient_metrics['patient_id'].nunique(),
        len(factor_table),
        int(patient_metrics.drop_duplicates(['chip', 'patient_id'])['n_neighborhoods'].sum()),
        len(patient_metrics),
        f"{validation['status'].eq('PASS').sum()}/{len(validation)}",
    ]
})
display(scope)
display(validation)

## Methods

- Factor usage and loadings came from the frozen per-chip Reference18 Neighborhood NMF K=6 models.
- Primary expression validation aggregated raw RNA over the same center-excluded 50 µm neighborhood used by NMF. Center-bin RNA was retained as independent focal evidence.
- Marker panels came from the six-dataset conservative Reference18 proposal.
- Marker-program associations were assessed within patient after adjustment for UMI, detected genes, and effective neighbor count.
- A complementary raw-count pseudobulk comparison matched high-usage bins to lower-usage bins on technical covariates.
- Patient metrics include continuous usage, balanced high-usage area, patient contribution, technical correlations, residual Moran's I, connected components, and local enrichment.
- Evidence labels summarize multiple axes and are not deletion gates or disease effects.

## Factor-level results

In [ ]:
evidence_counts = (factor_table['raw_expression_evidence']
                   .value_counts()
                   .rename_axis('Raw-expression evidence')
                   .reset_index(name='Factor count'))
display(evidence_counts)

factor_columns = [
    'chip', 'factor', 'top_state_1', 'top_state_2', 'top_state_3',
    'median_neighbor_composite_marker_residual_spearman',
    'median_composite_marker_log2fc',
    'positive_composite_marker_log2fc_patient_n',
    'patient_high_usage_coverage_n',
    'max_patient_mean_usage_contribution',
    'median_usage_umi_spearman', 'median_usage_n_genes_spearman',
    'raw_expression_evidence', 'final_evidence_level'
]
display(factor_table[factor_columns])

## B-associated factors

B-associated factors are retained as NMF results. This section distinguishes diffuse neighborhood-scale marker correlation from focal B-marker enrichment; weak continuous correlation does not by itself invalidate a localized B-rich factor.

In [ ]:
b_factors = factor_table.loc[factor_table['top_state_1'].eq('B'), [
    'chip', 'factor', 'median_neighbor_composite_marker_residual_spearman',
    'median_center_composite_marker_residual_spearman',
    'median_composite_marker_log2fc',
    'positive_composite_marker_log2fc_patient_n',
    'patient_high_usage_coverage_n', 'max_patient_mean_usage_contribution',
    'raw_expression_evidence', 'final_evidence_level'
]]
display(b_factors)

## English audit figures

In [ ]:
figure_names = [
    'FIG01_FACTOR_RAW_EXPRESSION_SUPPORT.png',
    'FIG02_EXPRESSION_SUPPORT_BY_PATIENT.png',
    'FIG03_FACTOR_PATIENT_LEVEL_METRICS.png',
    'FIG04_FACTOR_TECHNICAL_ASSOCIATIONS.png',
]
for figure_name in figure_names:
    display(Markdown(f'### {figure_name.removesuffix(".png").replace("_", " · ").title()}'))
    display(Image(filename=str(run_dir / 'figures' / figure_name), width=1100))

## Patient-level metrics

The complete 126-row table is shown below. Disease labels are metadata only; no disease comparison is performed in this report.

In [ ]:
display(patient_metrics)

## Marker provenance and coverage

In [ ]:
display(marker_audit)
display(manifest)

## Interpretation boundaries

The analysis validates whether frozen composition factors have corresponding raw-RNA and patient-level spatial evidence. It does not establish cell-cell interaction, causal niches, histologic identity, or disease association. Residual spatial structure is expected in overlapping neighborhoods and should not be interpreted as independent replication. Patient-specific factors remain valid candidates rather than automatic exclusions.

## Session information

In [ ]:
import platform
import sys
import numpy as np
import matplotlib

session = {
    'python': sys.version.replace('\n', ' '),
    'platform': platform.platform(),
    'pandas': pd.__version__,
    'numpy': np.__version__,
    'matplotlib': matplotlib.__version__,
    'analysis_output': str(run_dir),
}
display(pd.Series(session, name='value').to_frame())